In [ ]:
CATALOG = "spotify_etl"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
print("Target: spotify_etl.gold.gold_genre_trends")

In [ ]:
CATALOG = "spotify_etl"
SCHEMA = "gold"
TABLE = ""  # set per notebook

# Create shared base_analytics temp view
fct = spark.table(f"{CATALOG}.silver.fct_plays")
dim_t = spark.table(f"{CATALOG}.silver.dim_tracks")
dim_a = spark.table(f"{CATALOG}.silver.dim_artists")
dim_time = spark.table(f"{CATALOG}.silver.dim_time")

from pyspark.sql import functions as F
base = fct.join(dim_t, "track_id", "left") \
    .join(dim_a, fct.artist_id == dim_a.artist_id, "left") \
    .join(dim_time, "play_timestamp", "left") \
    .select(
        fct.play_id, fct.play_timestamp, fct.duration_ms, fct.context_type,
        dim_t.track_id, dim_t.track_name, dim_t.popularity,
        dim_a.artist_id, dim_a.artist_name, dim_a.genres,
        dim_time.hour_of_day, dim_time.weekday_name,
        F.date_format(fct.play_timestamp, "yyyy-MM").alias("month")
    )
base.createOrReplaceTempView("base_analytics")
print("Temp view 'base_analytics' created.")

In [ ]:
result = spark.sql('''
SELECT genre, month, COUNT(play_id) AS total_plays,
       AVG(popularity) AS avg_popularity, 1 AS unique_users
FROM base_analytics
LATERAL VIEW explode(genres) exploded_genres AS genre
WHERE genre IS NOT NULL
GROUP BY genre, month ORDER BY month, total_plays DESC
''')
full = f"{CATALOG}.{SCHEMA}.{TABLE}"
result.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full)
print(f"Table {full} written with {result.count()} rows.")

In [ ]:
result = spark.table(f"{CATALOG}.gold.gold_genre_trends")
print(f"Table state: {result.count()} rows")